# Beta-t-EGARCH dynamic scale modelling

This notebook estimates a score-driven Beta-t-EGARCH model for an equity-return series. The model allows conditional scale to respond to the score of a Student's *t* likelihood, so it can accommodate volatility clustering and heavy-tailed innovations.

**Workflow.** Load price data, form log returns, choose a scale specification, estimate it by maximum likelihood, and assess the fitted volatility and standardized residuals.


## 0. Preliminaries

The dynamic state is the log scale, \(\lambda_t = \log(\sigma_t)\). In the stationary specification used below, it evolves as

$$
\lambda_{t+1} = (1-\phi)\omega + \phi\lambda_t + \kappa u_t,
$$

with an optional asymmetric (leverage) term. The score \(u_t\) is derived from the conditional Student's *t* density. The helper functions `Beta_univ_t_Egarch_Logl` and `Beta_univ_t_Egarch_Logl_fact` must be in the current MATLAB folder or on the MATLAB path.


In [ ]:
clear all
close all


## 1. Load prices and construct returns

Select one price series by changing the active `xlsread` line. The notebook converts consecutive closing prices into log returns,

$$r_t = \log(P_t/P_{t-1}).$$

Ensure that the selected Excel file is in the working folder before running this cell.


In [ ]:
X_=xlsread('NSX Close---.xlsx'); %NSX

ss=size(X_,1);
X_1=X_([2:ss],:);
X_2=X_([1:ss-1],:);
X=log(X_1./X_2);


## 2. Configure the optimizers

Estimation is performed with `fmincon` when parameter bounds are active, or with `fminsearch` otherwise. These options control the numerical tolerances and iteration limits; tighter tolerances can improve convergence but may substantially increase run time.


In [ ]:
conoptions = optimoptions('fmincon','Algorithm','interior-point','Display','iter-detailed','MaxIter',1000*67,'MaxFunEvals',3000*67,'TolFun',0.00000001,'TolX',0.0000000000000001,'TolCon',0.00000000000001)
optionsunc = optimoptions('fminunc','Algorithm','quasi-newton','TolX',0.00001 ,'TolFun',0.00001, 'Display', 'iter-detailed', 'MaxIter', 1000*67, 'MaxFunEvals', 3000*67, 'HessUpdate', 'bfgs')

options = optimset('TolX',0.0000000000000001 ,'TolFun',0.00000001, 'Display', 'iter', 'MaxIter', 1000*67, 'MaxFunEvals', 3000*67, 'HessUpdate', 'bfgs')


## 3. Choose the scale specification

The active analysis estimates the dynamic-scale model. `I_s = 0` selects the mean-reverting AR(1) state equation; setting it to `1` selects an integrated/random-walk version. `Inf_s = 1` standardizes the score by its information, and `lev_s = 1` adds an asymmetric response to negative returns.

The degree-of-freedom and joint scale/degree-of-freedom switches are retained from the original research script for related extensions, but they are not used by the estimation steps below.


In [ ]:
I_s=0;
Inf_s=1;
lev_s=1;

Constr_s=1;

kk_s=0.01;


I_v=0;
Inf_v=1;
Stan=0;

I_sv_s=0;
Inf_sv_s=1;
lev_sv_s=0;
I_sv_v=0;
Inf_sv_v=1;

Constr_v=0;
Constr_sv=0;

Tst=1;

kk_v=0.9; %0.9 for standardized data after fitting Egarch scale
kk_sv=0.8; %0.7 better so far


## 4. Inspect the return series before estimation

These plots provide a quick check of the modelling motivation: the empirical distribution is compared with a Gaussian benchmark, while autocorrelations of returns and squared returns reveal serial dependence and volatility clustering.


In [ ]:
Y=X;

figure
[ff_x,r_x]=ecdf(Y);
ecdfhist(ff_x,r_x,100);
hold on
xx_x=min(Y):.001:max(Y);
yyn_x=1/sqrt(2*pi*std(Y)^2).*exp(-1/2*((xx_x-mean(Y))/std(Y)).^2);
plot(xx_x,yyn_x,'g-')
hold off
title('Unconditional Distr of the Data')

figure
autocorr(Y,100)
title('ACF Data')

figure
autocorr(Y.^2,100)
title('ACF Squared Data')

figure
qqplot(Y)
title('Empirical Density Data vs Gaussian Distr QQPlot')


## 5. Build starting values and parameter bounds

The parameter vector depends on the selected specification. For the stationary model with leverage it is

$$[\mu,\ \omega,\ \phi,\ \kappa,\ \log(\nu),\ \kappa_{\ell}]',$$

where \(\nu\) is the Student's *t* degrees of freedom. Starting values place the scale near the sample standard deviation and initialize \(\phi\) close to persistent, but stationary, dynamics. The bounds constrain the score coefficient to be non-negative and \(\phi\) to the interval \([0,1]\).


In [ ]:
t=size(Y,1);

if I_s==1
    
    if lev_s==1
        rS=ones(5,1)*kk_s;
        rS(1,1)=mean(Y);
        rS(2,1)=log(std(Y));
        rS(4,1)=log(8); %initial values of the df around 8
        
        if Constr_s==1
            lbS=-inf(5,1);
            ubS=inf(5,1);
            
            lbS(3,1)=0; %kapa dynamic score
        end
    
    else
        rS=ones(4,1)*kk_s;
        rS(1,1)=mean(Y);
        rS(2,1)=log(std(Y));
        rS(4,1)=log(8);

        if Constr_s==1
            lbS=-inf(4,1);
            ubS=inf(4,1);

            lbS(3,1)=0; %kapa dynamic score
        end
    
    end
    
else

    if lev_s==1
        rS=ones(6,1)*kk_s;
        rS(1,1)=mean(Y);
        rS(2,1)=log(std(Y));
        rS(3,1)=0.9;
        rS(5,1)=log(8);

        if Constr_s==1
            lbS=-inf(6,1);
            ubS=inf(6,1);

            ubS(3,1)=1; %phi stationary between -1 and 1
            lbS(3,1)=0; %set phi to be positive
            lbS(4,1)=0; %kapa dynamic score
        end
    
    else
        rS=ones(5,1)*kk_s;
        rS(1,1)=mean(Y);
        rS(2,1)=log(std(Y));
        rS(3,1)=0.9;
        rS(5,1)=log(8);

        if Constr_s==1
            lbS=-inf(5,1);
            ubS=inf(5,1);

            ubS(3,1)=1; %phi stationary between -1 and 1
            lbS(3,1)=0; %set phi to be positive 
            lbS(4,1)=0; %kapa dynamic score
        end
    
    end
    
end


## 6. Estimate the model by maximum likelihood

First evaluate the objective at the starting vector. Then minimize the negative log-likelihood using the selected optimizer. The fitted parameter vector is written to `estPar.xlsx`; the next call reconstructs the filtered scale, standardized residuals, and scores at the optimum.


In [ ]:
Beta_univ_t_Egarch_Logl (rS,Y,Inf_s,I_s)

f_s=@(x)Beta_univ_t_Egarch_Logl (x,Y,Inf_s,I_s)

if Constr_s==1
    [x_s,fval,exitflag,output,v_,w_,hessian] = fmincon(f_s,rS,[],[],[],[],lbS,ubS,[],conoptions)    
else
    [x_s,fval,exitflag,output] = fminsearch(f_s,rS,options)    
end

estPar=x_s;

xlswrite('estPar.xlsx',x_s);


[Logl_s,lam_s,res_s,u_s,Beta_s,fit_s]= Beta_univ_t_Egarch_Logl_fact (x_s,Y,Inf_s,I_s);
[BIC_s,AIC_s]=Info_Crit (Logl_s,x_s,t);


## 7. Recover interpretable parameter values

The optimizer works with \(\log(\nu)\), so this cell exponentiates it to recover the degrees of freedom. It also unpacks the estimated vector into named quantities, making the diagnostic plots below easier to interpret.


In [ ]:
Ysq=Y.^2;

par_s=x_s;

if I_s==1
    
    mu_s = par_s(1); %mean
    omega_s = par_s(2); %unconstrained mean lambda scale
    kapa_s= par_s(3); %dinamic cond score par
    vega_l_s = par_s(4); %log shape parameter
    
    if lev_s==1
        kapa_s_l=par_s(5);
    end
    
else

    mu_s = par_s(1); %mean
    omega_s = par_s(2); %unconstrained mean lambda scale
    phi_s = par_s(3); %dinamic AR parameter scale
    kapa_s= par_s(4); %dinamic cond score par
    vega_l_s = par_s(5); %log shape parameter
    
    if lev_s==1
        kapa_s_l=par_s(6);
    end
    
end

vega_s=exp(vega_l_s);


## 8. Compare returns with fitted conditional scale

The first panel compares demeaned returns with \(\hat\sigma_{t|t-1}=\exp(\hat\lambda_{t|t-1})\). The second uses absolute returns, which often makes the link between volatility bursts and the fitted scale more visible.


In [ ]:
figure
subplot(2,3,[1:3])
hold on
plot(Y-mu_s)
plot(exp(lam_s),'r')
box on
legend('$Y-\mu$ Returns','$\hat{\sigma}_{t|t-1}=\exp\left(\hat{\lambda}_{t|t-1}\right)$','Interpreter','latex')
title('Scale Fit - Returns')
subplot(2,3,[4:6])
hold on
plot(abs(Y-mu_s))
plot(exp(lam_s),'r')
box on
legend('$\left|Y-\mu\right|$ Returns','$\hat{\sigma}_{t|t-1}=\exp\left(\hat{\lambda}_{t|t-1}\right)$','Interpreter','latex')
title('Scale Fit - Abs Returns')


## 9. Check the standardized-residual distribution

Under a well-specified model, standardized residuals should follow the fitted Student's *t* distribution. This plot overlays the empirical density with both the fitted *t* density and the standard Gaussian density, helping assess the value added by heavy tails.


In [ ]:
figure
[ff_s,r_s]=ecdf(res_s);
ecdfhist(ff_s,r_s,100);
hold on
xx_s=min(res_s):.01:max(res_s);
yy_s=gamma((vega_s+1)/2)/(gamma(vega_s/2)*sqrt(pi*vega_s))*(ones(size(xx_s,1),1)+(xx_s.^2)/vega_s).^(-(vega_s+1)/2);
yyn_s=1/sqrt(2*pi).*exp(-1/2*xx_s.^2);
plot(xx_s,yy_s,'r-')
plot(xx_s,yyn_s,'g-')
legend('Empirical Density Residuals','Fitted t','Standard Gaussian')
hold off
box on
title('Empirical Density Fitted Residuals vs Fitted t Distr')


## 10. Diagnose remaining serial dependence

Residual autocorrelation indicates mean dynamics left unexplained; autocorrelation in squared residuals indicates remaining scale dynamics. The score ACF is also useful because persistent score dependence can signal that the state recursion needs refinement.


In [ ]:
figure
autocorr(res_s,100)
title('ACF Residuals')

figure
autocorr(res_s.^2,100)
title('ACF Squared Residuals')

figure
autocorr(u_s,100)
title('ACF Fitted Scores')


## 11. Assess calibration with the probability integral transform

The probability integral transform (PIT) maps each standardized residual through the fitted Student's *t* CDF. If the conditional distribution is correctly calibrated, sorted PIT values should closely follow the 45-degree uniform benchmark.


In [ ]:
fs=@(e)gamma((vega_s+1)/2)/(gamma(vega_s/2)*sqrt(pi*vega_s))*(1+(e.^2)/vega_s).^(-(vega_s+1)/2);
pit_s=@(g)integral(fs,-inf,g);
PIT_s=zeros(size(res_s,1),1);
ord_res_s=sort(res_s);
for i=1:size(res_s,1)
    PIT_s(i)=pit_s(ord_res_s(i));
end
PIT_uni_comp=linspace(0,1,size(PIT_s,1));

figure 
hold on
plot(PIT_s)
plot(PIT_uni_comp,'k--')
box on
title('Ordered PIT DCS Model vs Uniform')
xlabel('Residuals')
ylabel('PIT')
hold off
